In [90]:
from dotenv import load_dotenv

load_dotenv()

True

# Built-in tools

## DuckDuckGo Tool

In [2]:
from langchain_community.tools import DuckDuckGoSearchRun

In [9]:
search_tool= DuckDuckGoSearchRun()

result= search_tool.invoke('IPL')

print(result)

The 2025 Indian Premier League, also known asIPL18 and branded as TATAIPL2025, was the 18th edition of the Indian Premier League, a professional Twenty20 cricket league. The tournament featured 10 teams competing in 74 matches. It began on 22 March and was held across 13 venues before being suspended on 9 May due to the 2025 India-Pakistan crisis. The matches resumed from 17 May across ... Explore the latest updates, news, and insights about Indian Premier League. Stay informed with exclusive stories, match highlights, and detailed analysis related to Indian Premier League. Follow Indian Premier League onIPL.com for everything about cricket! IPL2025: Live scores, latest news, and exclusive player stats. See the full 2026 schedule and team standings here. Click for the latest updates! IPL2025: GetIPL2025 News on Crickit.. CheckIPLlatest updates,IPLMatch schedule, venue, Team, Player detailed stats along withIPLPoints table, fixtures and match results.IPLNews Today ... Dive into the worl

## Shell Tool

In [10]:
from langchain_community.tools import ShellTool

In [16]:
shell_tool= ShellTool()

result= shell_tool.invoke('whoami')

print(result)

Executing command:
 whoami
swayam\lenovo



# Custom Tools

## @tool decorator

In [18]:
from langchain.tools import tool

In [19]:
@tool
def multiply(a: int, b: int) -> int:
    """ Multiply two numbers. """
    return a * b

In [21]:
result= multiply.invoke({'a':2, 'b':5})
print(result)

10


## Structuring with Pydantic

In [23]:
from langchain.tools import tool
from pydantic import BaseModel, Field

In [25]:
class MultiplyInput(BaseModel):
    a: int= Field(description='The first number')
    b: int= Field(description='The second number')

In [26]:
@tool(args_schema= MultiplyInput)
def multiply(a: int, b: int) -> int:
    """ Multiply two numbers. """
    return a * b

In [27]:
result= multiply.invoke({'a':2, 'b':5})
print(result)

10


## BaseTool

In [ ]:
from langchain.tools import BaseTool
from pydantic import BaseModel
from typing import Type

In [29]:
class MultiplyInput(BaseModel):
    a: int= Field(description='The first number')
    b: int= Field(description='The second number')
    
class MultiplyTool(BaseTool):
    name: str= 'Multiply'
    description: str= 'Multiply two numbers'
    args_schema: Type[MultiplyInput]= MultiplyInput
    
    def _run(self, a: int, b: int) -> int:
        return a * b

In [30]:
multiply_tool= MultiplyTool()
result= multiply_tool.invoke({'a':2, 'b':5})
print(result)
print(multiply_tool.name)
print(multiply_tool.description)
print(multiply_tool.args_schema)

10
Multiply
Multiply two numbers
<class '__main__.MultiplyInput'>


# Tool Binding, Calling and Execution

In [ ]:
from langchain.tools import tool
from langchain_core.messages import HumanMessage
from langchain_mistralai import ChatMistralAI


In [75]:
@tool
def multiply(a: int, b: int) -> int:
    """ Given two numbers a and b, this tool returns their product. """
    return a * b

multiply.invoke({'a':2, 'b':5})

10

In [76]:
llm= ChatMistralAI(model='mistral-medium-latest')

In [77]:
llm_with_tool= llm.bind_tools([multiply])

In [78]:
result= llm_with_tool.invoke('How are you?')
print(result.content) # should contain a reply from the LLM
print(result.tool_calls) #should be empty as no tool was called

I'm just a program, so I don't have feelings, but I'm here and ready to help you! How about you? How are you doing?
[]


In [79]:
result= llm_with_tool.invoke('Can you multiply 2 by 5?')
print(result.content) # should be empty as tool call was made, so no response
print(result.tool_calls) #should contain the tool call


[{'name': 'multiply', 'args': {'a': 2, 'b': 5}, 'id': 'XRUdamP3b', 'type': 'tool_call'}]


In [80]:
result.tool_calls[0]

{'name': 'multiply',
 'args': {'a': 2, 'b': 5},
 'id': 'XRUdamP3b',
 'type': 'tool_call'}

In [81]:
multiply.invoke(result.tool_calls[0]['args'])

10

In [82]:
multiply.invoke({'name': 'multiply',
 'args': {'a': 2, 'b': 5},
 'id': 'pb8PBTfwS',
 'type': 'tool_call'})

# or multiply.invoke(result.tool_calls[0])

ToolMessage(content='10', name='multiply', tool_call_id='pb8PBTfwS')

In [83]:
query= HumanMessage('Can you multiply 2 by 5?')

messages= [query]
messages

[HumanMessage(content='Can you multiply 2 by 5?', additional_kwargs={}, response_metadata={})]

In [84]:
messages.append(result)
messages

[HumanMessage(content='Can you multiply 2 by 5?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'XRUdamP3b', 'function': {'name': 'multiply', 'arguments': '{"a": 2, "b": 5}'}, 'index': 0}]}, response_metadata={'token_usage': {'prompt_tokens': 89, 'total_tokens': 106, 'completion_tokens': 17, 'prompt_tokens_details': {'cached_tokens': 0}}, 'model_name': 'mistral-medium-latest', 'model': 'mistral-medium-latest', 'finish_reason': 'tool_calls', 'model_provider': 'mistralai'}, id='lc_run--019cc288-4ecc-7eb2-b10a-0d309eb28069-0', tool_calls=[{'name': 'multiply', 'args': {'a': 2, 'b': 5}, 'id': 'XRUdamP3b', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 89, 'output_tokens': 17, 'total_tokens': 106})]

In [85]:
tool_result= multiply.invoke(result.tool_calls[0])

messages.append(tool_result)
messages

[HumanMessage(content='Can you multiply 2 by 5?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'XRUdamP3b', 'function': {'name': 'multiply', 'arguments': '{"a": 2, "b": 5}'}, 'index': 0}]}, response_metadata={'token_usage': {'prompt_tokens': 89, 'total_tokens': 106, 'completion_tokens': 17, 'prompt_tokens_details': {'cached_tokens': 0}}, 'model_name': 'mistral-medium-latest', 'model': 'mistral-medium-latest', 'finish_reason': 'tool_calls', 'model_provider': 'mistralai'}, id='lc_run--019cc288-4ecc-7eb2-b10a-0d309eb28069-0', tool_calls=[{'name': 'multiply', 'args': {'a': 2, 'b': 5}, 'id': 'XRUdamP3b', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 89, 'output_tokens': 17, 'total_tokens': 106}),
 ToolMessage(content='10', name='multiply', tool_call_id='XRUdamP3b')]

In [ ]:
llm_with_tool.invoke(messages).content 

'The product of 2 and 5 is **10**.'

# Complete tool calling with api call

In [195]:
import os
from typing import Annotated

import requests
from langchain.tools import InjectedToolArg, tool
from langchain_core.messages import HumanMessage
from langchain_core.prompts import PromptTemplate
from langchain_mistralai import ChatMistralAI


In [241]:
@tool
def get_conversion_factor(base_currency: str, target_currency: str) -> float:
    """ 
    This function fetchers the currency rate conversion between given base and target currency.
    """
    EXCHANGE_RATE_API= os.getenv('EXCHANGE_RATE_API')
    url= f'https://v6.exchangerate-api.com/v6/{EXCHANGE_RATE_API}/pair/{base_currency}/{target_currency}'
    response= requests.get(url)
    
    return response.json()['conversion_rate']

@tool
def convert(base_currency: int, conversion_rate: Annotated[float, InjectedToolArg]) -> float:
    """ 
    Given a currency conversion rate, this function converts the given base currency to the target currency.
    """
    
    return base_currency * conversion_rate

In [242]:
get_conversion_factor.invoke({'base_currency': 'USD', 'target_currency': 'INR'})

91.8022

In [243]:
convert.invoke({'base_currency': 10, 'conversion_rate': 91.8022})

918.0219999999999

In [244]:
llm= ChatMistralAI(model='mistral-medium-latest')
llm_with_tools= llm.bind_tools([get_conversion_factor, convert])

In [245]:
llm_with_tools

RunnableBinding(bound=ChatMistralAI(profile={'max_input_tokens': 128000, 'max_output_tokens': 16384, 'image_inputs': True, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True}, client=<httpx.Client object at 0x00000274C85C2E70>, async_client=<httpx.AsyncClient object at 0x00000274C85C2D50>, mistral_api_key=SecretStr('**********'), endpoint='https://api.mistral.ai/v1', model='mistral-medium-latest', model_kwargs={}), kwargs={'tools': [{'type': 'function', 'function': {'name': 'get_conversion_factor', 'description': 'This function fetchers the currency rate conversion between given base and target currency.', 'parameters': {'properties': {'base_currency': {'type': 'string'}, 'target_currency': {'type': 'string'}}, 'required': ['base_currency', 'target_currency'], 'type': 'object'}}}, {'type': 'function', 'function': {'name': 'convert', 'description': 'Given a currency convers

In [246]:
messages=[HumanMessage('What is the conversion rate between USD and INR? Based on that, can you convert 10 USD to INR?')]
messages

[HumanMessage(content='What is the conversion rate between USD and INR? Based on that, can you convert 10 USD to INR?', additional_kwargs={}, response_metadata={})]

In [247]:
ai_message= llm_with_tools.invoke(messages)
ai_message.tool_calls

[{'name': 'get_conversion_factor',
  'args': {'base_currency': 'USD', 'target_currency': 'INR'},
  'id': 'QdtVKuQSP',
  'type': 'tool_call'}]

In [248]:
print(ai_message)

content='' additional_kwargs={'tool_calls': [{'id': 'QdtVKuQSP', 'function': {'name': 'get_conversion_factor', 'arguments': '{"base_currency": "USD", "target_currency": "INR"}'}, 'index': 0}]} response_metadata={'token_usage': {'prompt_tokens': 182, 'total_tokens': 204, 'completion_tokens': 22, 'prompt_tokens_details': {'cached_tokens': 0}}, 'model_name': 'mistral-medium-latest', 'model': 'mistral-medium-latest', 'finish_reason': 'tool_calls', 'model_provider': 'mistralai'} id='lc_run--019cc330-032c-7ac3-92ff-7c2752c4e951-0' tool_calls=[{'name': 'get_conversion_factor', 'args': {'base_currency': 'USD', 'target_currency': 'INR'}, 'id': 'QdtVKuQSP', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'input_tokens': 182, 'output_tokens': 22, 'total_tokens': 204}


In [249]:
messages.append(ai_message)
messages

[HumanMessage(content='What is the conversion rate between USD and INR? Based on that, can you convert 10 USD to INR?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'QdtVKuQSP', 'function': {'name': 'get_conversion_factor', 'arguments': '{"base_currency": "USD", "target_currency": "INR"}'}, 'index': 0}]}, response_metadata={'token_usage': {'prompt_tokens': 182, 'total_tokens': 204, 'completion_tokens': 22, 'prompt_tokens_details': {'cached_tokens': 0}}, 'model_name': 'mistral-medium-latest', 'model': 'mistral-medium-latest', 'finish_reason': 'tool_calls', 'model_provider': 'mistralai'}, id='lc_run--019cc330-032c-7ac3-92ff-7c2752c4e951-0', tool_calls=[{'name': 'get_conversion_factor', 'args': {'base_currency': 'USD', 'target_currency': 'INR'}, 'id': 'QdtVKuQSP', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 182, 'output_tokens': 22, 'total_tokens': 204})]

In [250]:
result= get_conversion_factor.invoke(ai_message.tool_calls[0])
conversion_factor= float(result.content)
result

ToolMessage(content='91.8022', name='get_conversion_factor', tool_call_id='QdtVKuQSP')

In [251]:
messages.append(result)
messages

[HumanMessage(content='What is the conversion rate between USD and INR? Based on that, can you convert 10 USD to INR?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'QdtVKuQSP', 'function': {'name': 'get_conversion_factor', 'arguments': '{"base_currency": "USD", "target_currency": "INR"}'}, 'index': 0}]}, response_metadata={'token_usage': {'prompt_tokens': 182, 'total_tokens': 204, 'completion_tokens': 22, 'prompt_tokens_details': {'cached_tokens': 0}}, 'model_name': 'mistral-medium-latest', 'model': 'mistral-medium-latest', 'finish_reason': 'tool_calls', 'model_provider': 'mistralai'}, id='lc_run--019cc330-032c-7ac3-92ff-7c2752c4e951-0', tool_calls=[{'name': 'get_conversion_factor', 'args': {'base_currency': 'USD', 'target_currency': 'INR'}, 'id': 'QdtVKuQSP', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 182, 'output_tokens': 22, 'total_tokens': 204}),
 ToolMessage(content='91.8022', name=

In [252]:
result= llm_with_tools.invoke(messages)
result

AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'ss06wJbtX', 'function': {'name': 'convert', 'arguments': '{"base_currency": 10}'}, 'index': 0}]}, response_metadata={'token_usage': {'prompt_tokens': 213, 'total_tokens': 225, 'completion_tokens': 12, 'prompt_tokens_details': {'cached_tokens': 0}}, 'model_name': 'mistral-medium-latest', 'model': 'mistral-medium-latest', 'finish_reason': 'tool_calls', 'model_provider': 'mistralai'}, id='lc_run--019cc330-14ef-7221-8241-29ffcfdec706-0', tool_calls=[{'name': 'convert', 'args': {'base_currency': 10}, 'id': 'ss06wJbtX', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 213, 'output_tokens': 12, 'total_tokens': 225})

In [253]:
messages.append(result)
messages

[HumanMessage(content='What is the conversion rate between USD and INR? Based on that, can you convert 10 USD to INR?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'QdtVKuQSP', 'function': {'name': 'get_conversion_factor', 'arguments': '{"base_currency": "USD", "target_currency": "INR"}'}, 'index': 0}]}, response_metadata={'token_usage': {'prompt_tokens': 182, 'total_tokens': 204, 'completion_tokens': 22, 'prompt_tokens_details': {'cached_tokens': 0}}, 'model_name': 'mistral-medium-latest', 'model': 'mistral-medium-latest', 'finish_reason': 'tool_calls', 'model_provider': 'mistralai'}, id='lc_run--019cc330-032c-7ac3-92ff-7c2752c4e951-0', tool_calls=[{'name': 'get_conversion_factor', 'args': {'base_currency': 'USD', 'target_currency': 'INR'}, 'id': 'QdtVKuQSP', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 182, 'output_tokens': 22, 'total_tokens': 204}),
 ToolMessage(content='91.8022', name=

In [254]:
result.tool_calls[0]

{'name': 'convert',
 'args': {'base_currency': 10},
 'id': 'ss06wJbtX',
 'type': 'tool_call'}

In [255]:
result.tool_calls[0]['args']['conversion_rate']= conversion_factor
result.tool_calls[0]

{'name': 'convert',
 'args': {'base_currency': 10, 'conversion_rate': 91.8022},
 'id': 'ss06wJbtX',
 'type': 'tool_call'}

In [256]:
result= convert.invoke(result.tool_calls[0])
result

ToolMessage(content='918.0219999999999', name='convert', tool_call_id='ss06wJbtX')

In [257]:
messages.append(result)
messages

[HumanMessage(content='What is the conversion rate between USD and INR? Based on that, can you convert 10 USD to INR?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'QdtVKuQSP', 'function': {'name': 'get_conversion_factor', 'arguments': '{"base_currency": "USD", "target_currency": "INR"}'}, 'index': 0}]}, response_metadata={'token_usage': {'prompt_tokens': 182, 'total_tokens': 204, 'completion_tokens': 22, 'prompt_tokens_details': {'cached_tokens': 0}}, 'model_name': 'mistral-medium-latest', 'model': 'mistral-medium-latest', 'finish_reason': 'tool_calls', 'model_provider': 'mistralai'}, id='lc_run--019cc330-032c-7ac3-92ff-7c2752c4e951-0', tool_calls=[{'name': 'get_conversion_factor', 'args': {'base_currency': 'USD', 'target_currency': 'INR'}, 'id': 'QdtVKuQSP', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 182, 'output_tokens': 22, 'total_tokens': 204}),
 ToolMessage(content='91.8022', name=

In [258]:
final_result= llm_with_tools.invoke(messages)
final_result

AIMessage(content='The current conversion rate between **USD and INR** is **1 USD = 91.80 INR**.\n\nBased on this rate, **10 USD** is equivalent to **918.02 INR**.', additional_kwargs={}, response_metadata={'token_usage': {'prompt_tokens': 258, 'total_tokens': 307, 'completion_tokens': 49, 'prompt_tokens_details': {'cached_tokens': 0}}, 'model_name': 'mistral-medium-latest', 'model': 'mistral-medium-latest', 'finish_reason': 'stop', 'model_provider': 'mistralai'}, id='lc_run--019cc330-da9f-7d32-b22e-de569dabe2cd-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 258, 'output_tokens': 49, 'total_tokens': 307})

## Manual Steps trial (These cells were executed before InjectedToolArg implementation)

In [107]:
ai_message= llm_with_tools.invoke(messages)
ai_message.tool_calls

[{'name': 'get_conversion_factor',
  'args': {'base_currency': 'USD', 'target_currency': 'INR'},
  'id': 'bdQe9o7bk',
  'type': 'tool_call'}]

In [ ]:
messages.append(ai_message)
messages

[HumanMessage(content='What is the conversion rate between USD and INR? Based on that, can you convert 10 USD to INR?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'bdQe9o7bk', 'function': {'name': 'get_conversion_factor', 'arguments': '{"base_currency": "USD", "target_currency": "INR"}'}, 'index': 0}]}, response_metadata={'token_usage': {'prompt_tokens': 198, 'total_tokens': 220, 'completion_tokens': 22, 'prompt_tokens_details': {'cached_tokens': 0}}, 'model_name': 'mistral-medium-latest', 'model': 'mistral-medium-latest', 'finish_reason': 'tool_calls', 'model_provider': 'mistralai'}, id='lc_run--019cc2b1-0e60-7a01-a0da-d30df6c6f743-0', tool_calls=[{'name': 'get_conversion_factor', 'args': {'base_currency': 'USD', 'target_currency': 'INR'}, 'id': 'bdQe9o7bk', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 198, 'output_tokens': 22, 'total_tokens': 220})]

In [111]:
result= get_conversion_factor.invoke(ai_message.tool_calls[0])
result

ToolMessage(content='{"result": "success", "documentation": "https://www.exchangerate-api.com/docs", "terms_of_use": "https://www.exchangerate-api.com/terms", "time_last_update_unix": 1772755201, "time_last_update_utc": "Fri, 06 Mar 2026 00:00:01 +0000", "time_next_update_unix": 1772841601, "time_next_update_utc": "Sat, 07 Mar 2026 00:00:01 +0000", "base_code": "USD", "target_code": "INR", "conversion_rate": 91.8022}', name='get_conversion_factor', tool_call_id='bdQe9o7bk')

In [112]:
messages.append(result)
messages

[HumanMessage(content='What is the conversion rate between USD and INR? Based on that, can you convert 10 USD to INR?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'bdQe9o7bk', 'function': {'name': 'get_conversion_factor', 'arguments': '{"base_currency": "USD", "target_currency": "INR"}'}, 'index': 0}]}, response_metadata={'token_usage': {'prompt_tokens': 198, 'total_tokens': 220, 'completion_tokens': 22, 'prompt_tokens_details': {'cached_tokens': 0}}, 'model_name': 'mistral-medium-latest', 'model': 'mistral-medium-latest', 'finish_reason': 'tool_calls', 'model_provider': 'mistralai'}, id='lc_run--019cc2b1-0e60-7a01-a0da-d30df6c6f743-0', tool_calls=[{'name': 'get_conversion_factor', 'args': {'base_currency': 'USD', 'target_currency': 'INR'}, 'id': 'bdQe9o7bk', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 198, 'output_tokens': 22, 'total_tokens': 220}),
 ToolMessage(content='{"result": "suc

In [113]:
intermediate_result= llm_with_tools.invoke(messages)
intermediate_result

AIMessage(content='', additional_kwargs={'tool_calls': [{'id': '5r5NVwAVm', 'function': {'name': 'convert', 'arguments': '{"base_currency": 10, "conversion_rate": 91.8022}'}, 'index': 0}]}, response_metadata={'token_usage': {'prompt_tokens': 398, 'total_tokens': 424, 'completion_tokens': 26, 'prompt_tokens_details': {'cached_tokens': 0}}, 'model_name': 'mistral-medium-latest', 'model': 'mistral-medium-latest', 'finish_reason': 'tool_calls', 'model_provider': 'mistralai'}, id='lc_run--019cc2c3-8618-7673-9625-71fd9d576ff0-0', tool_calls=[{'name': 'convert', 'args': {'base_currency': 10, 'conversion_rate': 91.8022}, 'id': '5r5NVwAVm', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 398, 'output_tokens': 26, 'total_tokens': 424})

In [115]:
messages.append(intermediate_result)
messages

[HumanMessage(content='What is the conversion rate between USD and INR? Based on that, can you convert 10 USD to INR?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'bdQe9o7bk', 'function': {'name': 'get_conversion_factor', 'arguments': '{"base_currency": "USD", "target_currency": "INR"}'}, 'index': 0}]}, response_metadata={'token_usage': {'prompt_tokens': 198, 'total_tokens': 220, 'completion_tokens': 22, 'prompt_tokens_details': {'cached_tokens': 0}}, 'model_name': 'mistral-medium-latest', 'model': 'mistral-medium-latest', 'finish_reason': 'tool_calls', 'model_provider': 'mistralai'}, id='lc_run--019cc2b1-0e60-7a01-a0da-d30df6c6f743-0', tool_calls=[{'name': 'get_conversion_factor', 'args': {'base_currency': 'USD', 'target_currency': 'INR'}, 'id': 'bdQe9o7bk', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 198, 'output_tokens': 22, 'total_tokens': 220}),
 ToolMessage(content='{"result": "suc

In [116]:
intermediate_result_again= convert.invoke(intermediate_result.tool_calls[0])
intermediate_result_again

ToolMessage(content='918.0219999999999', name='convert', tool_call_id='5r5NVwAVm')

In [117]:
messages.append(intermediate_result_again)
messages

[HumanMessage(content='What is the conversion rate between USD and INR? Based on that, can you convert 10 USD to INR?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'bdQe9o7bk', 'function': {'name': 'get_conversion_factor', 'arguments': '{"base_currency": "USD", "target_currency": "INR"}'}, 'index': 0}]}, response_metadata={'token_usage': {'prompt_tokens': 198, 'total_tokens': 220, 'completion_tokens': 22, 'prompt_tokens_details': {'cached_tokens': 0}}, 'model_name': 'mistral-medium-latest', 'model': 'mistral-medium-latest', 'finish_reason': 'tool_calls', 'model_provider': 'mistralai'}, id='lc_run--019cc2b1-0e60-7a01-a0da-d30df6c6f743-0', tool_calls=[{'name': 'get_conversion_factor', 'args': {'base_currency': 'USD', 'target_currency': 'INR'}, 'id': 'bdQe9o7bk', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 198, 'output_tokens': 22, 'total_tokens': 220}),
 ToolMessage(content='{"result": "suc

In [118]:
final_result= llm_with_tools.invoke(messages)
final_result

AIMessage(content='The current conversion rate between **USD and INR** is **1 USD = 91.8022 INR**.\n\nBased on this rate, **10 USD** is equivalent to **918.02 INR**.', additional_kwargs={}, response_metadata={'token_usage': {'prompt_tokens': 443, 'total_tokens': 494, 'completion_tokens': 51, 'prompt_tokens_details': {'cached_tokens': 0}}, 'model_name': 'mistral-medium-latest', 'model': 'mistral-medium-latest', 'finish_reason': 'stop', 'model_provider': 'mistralai'}, id='lc_run--019cc2c5-4848-7900-9b9b-3cb24fb190ee-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 443, 'output_tokens': 51, 'total_tokens': 494})

In [119]:
final_result.content

'The current conversion rate between **USD and INR** is **1 USD = 91.8022 INR**.\n\nBased on this rate, **10 USD** is equivalent to **918.02 INR**.'